In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.llms import Ollama
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

### **Terraria chatbot**
This code is reponsible for utalizing a vector base created in other notebook. It loads vector base, sets up language model with a given context and allows user to test prompts

**Loading embeddings model**

In [6]:
embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


**Connecting to vector base**

In [7]:
vector_base = Chroma(
    persist_directory="./chroma_terraria_db",
    embedding_function=embeddings
)

**Setting up language model**

In [8]:
llm = Ollama(model='phi3')

system_prompt = (
    "You are an expert Terraria assistant. Your ONLY job is to extract answers from the provided CONTEXT.\n"
    "CRITICAL RULES:\n"
    "1. If the answer is not explicitly written in the CONTEXT below, you MUST reply exactly with: 'I cannot answer this based on the provided text.'\n"
    "2. Do NOT invent items, bosses, or mechanics. Do NOT use your outside knowledge.\n"
    "3. Base your answer ONLY on the text below.\n\n"
    "CONTEXT:\n{context}"
)

# system_prompt = (
#     "Answer based on given source\n"
#     "CRITICAL RULES:\n"
#     "Make no mistakes\n"
# )

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

retriever = vector_base.as_retriever(search_kwargs={"k": 4})
qa_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, qa_chain)

**Generating an example answer**

In [9]:
question = "Tell me about trees in Terraria"
print(f"Question: {question}")
print("Generating an answer")

answer = rag_chain.invoke({"input": question})
print("\nAI answer:")
print(answer['answer'])

Question: Tell me about trees in Terraria
Generating an answer

AI answer:
In Terraria, there are various types of trees that grow within different biomes such as Forests (Purity/Forest), Corruption, Crimson, Hallow, Snow Mesa and Oasis. Trees typically appear at the surface or near-surface with a similar height in clusters when they naturally spawn in these environments.

Trees can be chopped down using an axe or chainsaw to obtain Wood which is vital for crafting within Terraria's base game mechanics, as well as Shadewood and Pearlwood trees from Forest biomes that are introduced later on with additional functionalities including Acorn drops when harvested. Destroying a tree through the use of explosives or lava will yield units of Wood based on its size; larger trees provide more resources upon destruction, as well as potential chances to obtain one or more Acorns which can be used for planting and growing new trees.

Certain unique varieties exist such as Shadewood from Crimson bio

Tests

In [10]:
found_pieces = vector_base.similarity_search(question, k=4)

print("What base found:")
for i, piece in enumerate(found_pieces):
    print(f"\nResult {i+1} (article: {piece.metadata.get('title', 'No title')}):")
    print(piece.page_content[:150] + "...")

What base found:

Result 1 (article: Trees):
This is the main page whose information applies to the Desktop , Console , and Mobile versions of Terraria . For the differences of this information o...

Result 2 (article: Shadewood):
This is the main page whose information applies to the Desktop , Console , and Mobile versions of Terraria . For the differences of this information o...

Result 3 (article: Hexxed Branch):
is in water, despite the fact that in real life trees rarely move at all. This is potentially a reference to the Huorns from J. R. R. Tolkien's The Lo...

Result 4 (article: Trees):
parts of the tree, but cutting at the lowermost center tile will destroy the entire tree. If parts of the tree are left, they will not regrow. Blocks ...
